In [2]:
import subprocess

result = subprocess.run(
    ["python", "-m", "src.retinocare.models.train",
     "--config", "configs/train_config.yaml",
     "--model", "resnet18"],
    cwd="..", capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)

Using device: cpu
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\user/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth
Epoch 1/30 | train_loss=1.5292 train_f1=0.4875 | val_loss=1.4082 val_f1=0.5571
  -> new best val_f1=0.5571, saved to models\resnet18.pt
Epoch 2/30 | train_loss=1.3270 train_f1=0.6270 | val_loss=1.2601 val_f1=0.6168
  -> new best val_f1=0.6168, saved to models\resnet18.pt
Epoch 3/30 | train_loss=1.2444 train_f1=0.6431 | val_loss=1.1728 val_f1=0.6473
  -> new best val_f1=0.6473, saved to models\resnet18.pt
Epoch 4/30 | train_loss=1.1626 train_f1=0.6762 | val_loss=1.1466 val_f1=0.6416
Epoch 5/30 | train_loss=1.1356 train_f1=0.6742 | val_loss=1.0981 val_f1=0.6255
Epoch 5: unfreezing backbone for full fine-tuning
Epoch 6/30 | train_loss=0.9665 train_f1=0.7008 | val_loss=0.9666 val_f1=0.7270
  -> new best val_f1=0.7270, saved to models\resnet18.pt
Epoch 7/30 | train_loss=0.7890 train_f1=0.7745 | val_loss=0.9298 val_f1=0.7488
  -> new b

In [4]:
import sys
sys.path.append("..")

import yaml, torch
import torch.nn.functional as F
from src.retinocare.data.dataset import get_dataloaders
from src.retinocare.models.resnet_transfer import build_resnet18
from src.retinocare.evaluation.metrics import compute_classification_report, plot_confusion_matrix, expected_calibration_error

with open("../configs/train_config.yaml") as f:
    config = yaml.safe_load(f)

config["data"]["train_csv"] = "../" + config["data"]["train_csv"]
config["data"]["val_csv"] = "../" + config["data"]["val_csv"]
config["data"]["test_csv"] = "../" + config["data"]["test_csv"]
config["data"]["image_dir"] = "../" + config["data"]["image_dir"]

_, _, test_dl = get_dataloaders(config)

model = build_resnet18(num_classes=5, freeze_backbone=False)
model.load_state_dict(torch.load("../models/resnet18.pt", map_location="cpu"))
model.eval()

all_preds, all_labels, all_probs = [], [], []
with torch.no_grad():
    for images, labels in test_dl:
        logits = model(images)
        probs = F.softmax(logits, dim=1)
        all_preds.extend(logits.argmax(dim=1).tolist())
        all_labels.extend(labels.tolist())
        all_probs.extend(probs.tolist())

import numpy as np
all_probs = np.array(all_probs)

SEVERITY_LABELS = ["No DR", "Mild", "Moderate", "Severe", "Proliferative DR"]
report = compute_classification_report(all_labels, all_preds, SEVERITY_LABELS)
print("Weighted F1:", report["weighted avg"]["f1-score"])

plot_confusion_matrix(all_labels, all_preds, SEVERITY_LABELS, "../docs/resnet18_confusion_matrix.png")

ece = expected_calibration_error(all_probs, all_labels, n_bins=10)
print(f"Expected Calibration Error: {ece:.4f}")

c:\ProgramData\anaconda3\envs\retinocare-ai\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Weighted F1: 0.7951825885836976
Expected Calibration Error: 0.0547


# ResNet18 Training Results

## Setup
- Architecture: ResNet18 (ImageNet-pretrained), two-stage fine-tuning
- Stage 1 (epochs 1–5): backbone frozen, head-only training
- Stage 2 (epoch 6 onward): full network unfrozen, fine-tuned at 10x lower LR
- Loss: CrossEntropyLoss with inverse-frequency class weighting
- Early stopping: patience=5 on validation weighted F1

## Training trajectory
- Frozen-head stage plateaued around val F1 ≈ 0.63–0.65
- Unfreezing the backbone (epoch 6) produced an immediate jump to val F1 = 0.727,
  confirming the two-stage approach meaningfully outperforms head-only training
- Best checkpoint: epoch 13, val F1 = 0.7797
- Training stopped early at epoch 21 (5 epochs with no val improvement);
  train F1 had continued climbing to 0.92+ while val F1 plateaued — expected
  overfitting signal, correctly caught by early stopping

## Test set results
- **Weighted F1: 0.7952**
- **Expected Calibration Error (ECE): 0.0547** — well-calibrated; confidence
  scores can be reasonably trusted by the downstream RAG agent

## Per-class performance (derived from confusion matrix)

| Severity          | Recall | Precision (approx.) | Support |
|-------------------|--------|----------------------|---------|
| No DR             | 96.7%  | 97.0%                | 271     |
| Mild              | 53.6%  | 57.7%                | 56      |
| Moderate          | 72.7%  | 74.7%                | 150     |
| **Severe**        | **37.9%** | **30.6%**         | 29      |
| Proliferative DR  | 54.5%  | 52.2%                | 44      |

## Key finding

The model's weakest performance is on the **Severe** class, which is both the
rarest class in the test set and the most clinically consequential to miss
(under-classifying Severe as Moderate could delay an urgent referral — see
`knowledge_base/guidelines/referral-criteria.md`). Most Severe misclassifications
went to either Moderate or Proliferative DR, i.e. the model rarely confuses
Severe with the far end of the scale (No DR/Mild), but does struggle to draw
the line between adjacent severity levels.



In [5]:
result = subprocess.run(
    ["python", "-m", "src.retinocare.models.train",
     "--config", "configs/train_config.yaml",
     "--model", "efficientnet_b0"],
    cwd="..", capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)

Using device: cpu
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to C:\Users\user/.cache\torch\hub\checkpoints\efficientnet_b0_rwightman-7f5810bc.pth
Epoch 1/30 | train_loss=1.4893 train_f1=0.5024 | val_loss=1.3421 val_f1=0.5886
  -> new best val_f1=0.5886, saved to models\efficientnet_b0.pt
Epoch 2/30 | train_loss=1.2978 train_f1=0.6293 | val_loss=1.2404 val_f1=0.6389
  -> new best val_f1=0.6389, saved to models\efficientnet_b0.pt
Epoch 3/30 | train_loss=1.1991 train_f1=0.6480 | val_loss=1.1761 val_f1=0.6717
  -> new best val_f1=0.6717, saved to models\efficientnet_b0.pt
Epoch 4/30 | train_loss=1.1410 train_f1=0.6623 | val_loss=1.1388 val_f1=0.6589
Epoch 5/30 | train_loss=1.1152 train_f1=0.6765 | val_loss=1.1184 val_f1=0.6735
  -> new best val_f1=0.6735, saved to models\efficientnet_b0.pt
Epoch 5: unfreezing backbone for full fine-tuning
Epoch 6/30 | train_loss=1.0350 train_f1=0.6930 | val_loss=1.0304 val_f1=0.7023
  -> new best val_f1=0.7023

In [6]:
import sys
sys.path.append("..")

import yaml, torch
import torch.nn.functional as F
from src.retinocare.data.dataset import get_dataloaders
from src.retinocare.models.resnet_transfer import build_efficientnet_b0
from src.retinocare.evaluation.metrics import compute_classification_report, plot_confusion_matrix, expected_calibration_error

with open("../configs/train_config.yaml") as f:
    config = yaml.safe_load(f)

config["data"]["train_csv"] = "../" + config["data"]["train_csv"]
config["data"]["val_csv"] = "../" + config["data"]["val_csv"]
config["data"]["test_csv"] = "../" + config["data"]["test_csv"]
config["data"]["image_dir"] = "../" + config["data"]["image_dir"]

_, _, test_dl = get_dataloaders(config)

model = build_efficientnet_b0(num_classes=5, freeze_backbone=False)
model.load_state_dict(torch.load("../models/efficientnet_b0.pt", map_location="cpu"))
model.eval()

all_preds, all_labels, all_probs = [], [], []
with torch.no_grad():
    for images, labels in test_dl:
        logits = model(images)
        probs = F.softmax(logits, dim=1)
        all_preds.extend(logits.argmax(dim=1).tolist())
        all_labels.extend(labels.tolist())
        all_probs.extend(probs.tolist())

import numpy as np
all_probs = np.array(all_probs)

SEVERITY_LABELS = ["No DR", "Mild", "Moderate", "Severe", "Proliferative DR"]
report = compute_classification_report(all_labels, all_preds, SEVERITY_LABELS)
print("Weighted F1:", report["weighted avg"]["f1-score"])

plot_confusion_matrix(all_labels, all_preds, SEVERITY_LABELS, "../docs/efficientnet_b0_confusion_matrix.png")

ece = expected_calibration_error(all_probs, all_labels, n_bins=10)
print(f"Expected Calibration Error: {ece:.4f}")

Weighted F1: 0.8133297936980781
Expected Calibration Error: 0.0966


# EfficientNet-B0 Training Results

## Setup
- Architecture: EfficientNet-B0 (ImageNet-pretrained), two-stage fine-tuning
  (identical protocol to ResNet18: frozen head for 5 epochs, then full
  fine-tuning at 10x lower LR)
- Loss: CrossEntropyLoss with inverse-frequency class weighting
- Early stopping: patience=5 on validation weighted F1

## Training trajectory
- Frozen-head stage: val F1 climbed steadily 0.589 → 0.674 (smoother than
  ResNet18's equivalent stage)
- Unfreezing at epoch 5 produced a smaller immediate jump than ResNet18 saw
  (0.674 → 0.702, vs ResNet18's 0.626 → 0.727), but continued improving more
  steadily afterward
- Best checkpoint: epoch 30 (ran the full 30 epochs without early stopping
  triggering, unlike both other models) — val F1 = 0.7940
- Train F1 reached 0.91 by the end, similar overfitting gap to ResNet18

## Test set results
- **Weighted F1: 0.8133** — highest of the three models
- **Expected Calibration Error (ECE): 0.0966** — noticeably worse calibrated
  than ResNet18 (0.0547); the model is more often overconfident relative to
  its actual accuracy, which matters since the RAG agent phrases its
  recommendation based on this confidence score

## Per-class performance (derived from confusion matrix)

| Severity          | Recall | Precision | Support |
|-------------------|--------|-----------|---------|
| No DR             | 97.0%  | 96.7%     | 271     |
| Mild              | 58.9%  | 55.0%     | 56      |
| Moderate          | 75.3%  | 80.7%     | 150     |
| Severe            | 44.8%  | 37.1%     | 29      |
| Proliferative DR  | 54.5%  | 55.8%     | 44      |

## Key finding

EfficientNet-B0 achieves the best raw weighted F1 and the best Severe-class
recall (44.8%, vs ResNet18's 37.9% and baseline's 20.7%) of all three models.
However, its calibration is worse than ResNet18's — meaning its confidence
scores are somewhat less trustworthy as a signal for the downstream agent.
This is a genuine trade-off, not a clear win: **higher raw accuracy vs.
more trustworthy confidence scores**, and the right choice depends on
whether the agent layer leans more on the predicted label or the
confidence value when phrasing its recommendation.